In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

import pandas as pd
import matplotlib.pyplot as plt
import os
csv_path = os.path.join(path, "Q1_data.csv")

data = pd.read_csv(csv_path)




In [ ]:
# Task 2: Write your code here:
print("First 5 rows of the dataset:")
print(data.head())

In [ ]:
# Task 3: Write your code here:
print("\nDataset info:")
print(data.info())


In [ ]:
# Task 4: Write your code here:
print("\nStatistical description:")
print(data.describe())


In [ ]:
# Task 5: Write your code here:


target_col = [col for col in data.columns if 'delivery' in col.lower() and 'time' in col.lower()]
if len(target_col) == 0:
    raise ValueError("No column matching 'delivery_time' found.")
target_col = target_col[0]
print(f"\nDetected target column: {target_col}")

plt.figure(figsize=(8,5))
plt.hist(data[target_col], bins=30, color='skyblue', edgecolor='black')
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()



In [ ]:
# Task 1: Write your code here:

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
if 'Order_ID' in data.columns:
    data = data.drop(columns=['Order_ID'])
    print("Dropped 'Order_ID' column.")



In [ ]:
# Task 2: Write your code here:
missing_percentage = (data.isnull().sum() / len(data)) * 100
missing_data = missing_percentage[missing_percentage > 0].sort_values(ascending=False)
print("\nColumns with missing values (%):")
print(missing_data)

numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].mean())


categorical_cols = data.select_dtypes(include=['object']).columns
for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])




In [ ]:
# Task 3: Write your code here:
duplicates = data.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")
if duplicates > 0:
    data = data.drop_duplicates()
    print("Duplicates removed.")


In [ ]:
# Task 4: Write your code here:
categorical_cols = data.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)
    print(f"Applied One Hot Encoding to columns: {list(categorical_cols)}")


In [ ]:
# Task 5: Write your code here:
target_col = [col for col in data.columns if 'delivery' in col.lower() and 'time' in col.lower()][0]
features = data.drop(columns=[target_col])
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
data_scaled = pd.DataFrame(features_scaled, columns=features.columns)
data_scaled[target_col] = data[target_col].values


In [ ]:
# Task 6: Write your code here:
print("\nData cleaning and preprocessing complete.")
print("Scaled data shape:", data_scaled.shape)

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

target_col = [col for col in data_scaled.columns if 'delivery' in col.lower() and 'time' in col.lower()][0]
X = data_scaled.drop(columns=[target_col])
y = data_scaled[target_col]


In [ ]:
# Task 2,3,4,5: Write your code here:
kf = KFold(n_splits=5, shuffle=True, random_state=42)  # Use shuffle=True for better distribution

mae_scores = []

for fold, (train_index, valid_index) in enumerate(kf.split(X), 1):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)

    y_pred = rf_model.predict(X_valid)
    mae = mean_absolute_error(y_valid, y_pred)
    mae_scores.append(mae)
    print(f"Fold {fold} MAE: {mae:.3f}")

print(f"\nAverage MAE across all folds: {np.mean(mae_scores):.3f}")


In [ ]:
# Task 1: Write your code here:
feature_importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values(by='importance', ascending=False)


top_features = feature_importances.head(20).sort_values(by='importance')

plt.figure(figsize=(10,8))
plt.barh(top_features['feature'], top_features['importance'], color='skyblue')
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances from RandomForest')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:
y_pred_full = rf_model.predict(X)

plt.figure(figsize=(8,5))
plt.hist(y_pred_full, bins=30, color='lightcoral', edgecolor='black')
plt.title('Histogram of Predicted Delivery Time')
plt.xlabel('Predicted Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here:

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np
target_col = [col for col in data_scaled.columns if 'delivery' in col.lower() and 'time' in col.lower()][0]
X = data_scaled.drop(columns=[target_col])
y = data_scaled[target_col]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []
for fold, (train_index, valid_index) in enumerate(kf.split(X), 1):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    cb_model.fit(X_train, y_train)

    y_pred_rf = rf_model.predict(X_valid)
    y_pred_cb = cb_model.predict(X_valid)

    y_pred_ensemble = (y_pred_rf + y_pred_cb) / 2

    mae = mean_absolute_error(y_valid, y_pred_ensemble)
    mae_scores.append(mae)
    print(f"Fold {fold} Ensemble MAE: {mae:.3f}")

print(f"\nAverage Ensemble MAE across all folds: {np.mean(mae_scores):.3f}")


